# TabuLLM: Fraud Detection Walkthrough

This notebook demonstrates the core **TabuLLM** workflow on a fraud detection dataset:

- **Embedding** text columns with `TextColumnTransformer`
- **Dimensionality reduction** with `GMMFeatureExtractor`
- **Cluster explanation** with `ClusterExplainer`
- **Predictive pipeline** combining text and structured features

A TF-IDF baseline is included to motivate why semantic embeddings matter.

**Requirements:**
- `pip install tabullm[examples]`
- The embedding step uses `sentence-transformers/all-MiniLM-L6-v2` (free, runs locally).
- The `ClusterExplainer` step calls an LLM. Set `LLM_PROVIDER` below to `"openai"` or `"bedrock"`,
  and provide the corresponding credentials in a `.env` file or environment variables. See `.env.example` for details.
- All ClusterExplainer outputs are pre-computed in this notebook.
  Set `RERUN = True` below to re-execute the LLM calls.

In [ ]:
# --- Configuration ---
RERUN = False          # Set to True to re-execute LLM calls
LLM_PROVIDER = "bedrock"  # "openai" or "bedrock"

In [2]:
# Load environment variables from .env file (API keys, AWS credentials, etc.)
# override=True ensures .env values take precedence over system env vars
from dotenv import load_dotenv
load_dotenv(override=True)

True

## 1. Loading the Data

The fraud detection dataset consists of 17,880 job postings collected from employment websites,
each labeled as fraudulent (4.8%) or legitimate. It contains seven text columns capturing
different aspects of each posting — job title, geographic location, department, company profile,
full description, requirements, and benefits — alongside eight structured features (three binary,
five categorical). This dataset exemplifies the core TabuLLM use case: tabular data where text
columns provide semantic information that complements structured features. The dataset is bundled
with the package and is downloaded automatically from Zenodo on first use via `load_fraud()`; no
credentials are required.

In [4]:
from tabullm import load_fraud

X, y, metadata = load_fraud()

print(f"Samples: {metadata['n_samples']}")
print(f"Fraud rate: {metadata['class_distribution']['fraud_rate']:.1%}")
print(f"\nColumns ({len(X.columns)}):")
print(f"  Text (7): {metadata['text_columns']}")
print(f"  Binary (3): {metadata['binary_columns']}")
print(f"  Categorical (5): {metadata['categorical_columns']}")

/Users/amahani/code/TabuLLM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Samples: 17880
Fraud rate: 4.8%

Columns (15):
  Text (7): ['title', 'location', 'department', 'company_profile', 'description', 'requirements', 'benefits']
  Binary (3): ['telecommuting', 'has_company_logo', 'has_questions']
  Categorical (5): ['employment_type', 'required_experience', 'required_education', 'industry', 'function']


In [5]:
text_cols = metadata['text_columns']
X[text_cols].head(2)

,title,location,department,company_profile,description,requirements,benefits
0,Marketing Intern,"US, NY, New York",Marketing,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...


## 2. Traditional Vectorization (TF-IDF)

scikit-learn's `TfidfVectorizer` produces sparse bag-of-words representations based on term
frequencies. While computationally efficient, these features capture lexical patterns (which
words appear) rather than semantic meaning (what the text means). Two job postings using
different vocabularies to describe the same role ("Software Engineer" versus "Code Developer")
would appear dissimilar despite semantic equivalence.

For datasets with multiple text columns, manual concatenation is required:

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Manual concatenation of all 7 text columns
texts = X[text_cols].fillna('').apply(
    lambda row: ' || '.join(row), axis=1
)

# Vectorize
vectorizer = TfidfVectorizer(max_features=384)
X_tfidf = vectorizer.fit_transform(texts).toarray()

print(f"TF-IDF features: {X_tfidf.shape}")

TF-IDF features: (17880, 384)


## 3. Embedding Text Columns with `TextColumnTransformer`

`TextColumnTransformer` addresses both the semantic limitation of TF-IDF (by providing access
to LLM embeddings that capture meaning) and the integration challenge (through native
multi-column handling within scikit-learn pipelines).

It wraps any LangChain-supported embedding model in a scikit-learn `fit`/`transform` interface:

In [ ]:
from tabullm import TextColumnTransformer
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
)

text_transformer = TextColumnTransformer(model=embedding_model)
X_emb = text_transformer.fit_transform(X[text_cols])

print(f"LLM embeddings shape: {X_emb.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5064.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM embeddings shape: (17880, 384)


The result is a 17,880 x 384 dense matrix of semantic vectors. Compare the API to the TF-IDF
approach above: no manual concatenation, no `.fillna()`, no `.toarray()` — and the embeddings
capture semantic meaning rather than lexical overlap.

However, 384 dimensions are high-dimensional and opaque — motivating dimensionality reduction.

## 4. Dimensionality Reduction with `GMMFeatureExtractor`

`GMMFeatureExtractor` fits a Gaussian mixture model and returns per-cluster log-joint features,
reducing the embedding space from hundreds of dimensions to K interpretable features:

In [9]:
from tabullm import GMMFeatureExtractor

gmm = GMMFeatureExtractor(n_components=10, random_state=42)
X_features = gmm.fit_transform(X_emb)
cluster_labels = gmm.labels_

print(f"GMM log-joint features shape: {X_features.shape}")
print(f"Reduction: {X_emb.shape[1]}D -> {X_features.shape[1]}D")

GMM log-joint features shape: (17880, 10)
Reduction: 384D -> 10D


The 384-dimensional embedding is reduced to 10 log-joint features — one per GMM component.
Setting `include_log_density=True` appends the marginal log-density as an 11th feature,
which serves as an explicit outlier score.

### Cluster quality diagnostics

The `assignment_confidence_stats()` method returns per-observation diagnostics characterizing
how confidently each observation belongs to its assigned cluster:

In [10]:
obs_stats = gmm.assignment_confidence_stats(X_emb)
obs_stats.head()

,max_posterior,entropy,log_joint_margin,log_density
0,1.0,8.400815e-194,447.709797,1092.544703
1,1.0,8.907784e-259,597.319231,1134.216497
2,1.0,1.316880e-123,286.079318,1038.792000
3,1.0,3.885767e-154,356.377402,1052.550489
4,1.0,8.215763e-141,325.695061,1126.149399


The four columns are:
- **max_posterior**: highest posterior probability across clusters (1/K to 1)
- **entropy**: Shannon entropy of the posterior distribution (high = uncertain)
- **log_joint_margin**: gap between the top-two cluster log-joints (high = well-separated)
- **log_density**: marginal log-likelihood (low = outlier)

These diagnostics can be passed directly to `ClusterExplainer` for outcome-based analysis.

## 5. Cluster Explanation with `ClusterExplainer`

`ClusterExplainer` generates natural language descriptions of each cluster using LLMs,
with optional outcome-based statistical testing and narrative synthesis.

### Cost preview

Before making LLM calls, the `preview` flag estimates token usage and cost:

In [19]:
from tabullm import ClusterExplainer

# Initialize LLM based on provider choice
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    llm_cost = 0.15  # USD per 1M input tokens
elif LLM_PROVIDER == "bedrock":
    from langchain_aws import ChatBedrock
    llm = ChatBedrock(
        model_id='us.anthropic.claude-haiku-4-5-20251001-v1:0',
        region_name='us-east-1',
        model_kwargs={"max_tokens": 4096, "temperature": 0.0},
    )
    llm_cost = 1.00
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}")

print(f"Using LLM provider: {LLM_PROVIDER}")

explainer = ClusterExplainer(
    llm=llm,
    text_transformer=text_transformer,
    observations='job postings',
    text_fields='title, location, department, company profile, '
               'description, requirements, and benefits',
    llm_cost_per_1M_tokens=llm_cost,
)

_ = explainer.explain(
    X, cluster_labels,
    max_members_per_cluster=100,
    random_state=42,
    preview=True
)

Using LLM provider: bedrock
Preview
───────
Tokens (estimated):   699,328
Cost (estimated):     $0.6993
Strategy:             two-stage (auto-selected)
Max members/cluster:  100



### Full explanation

The full call generates cluster descriptions (blind to the outcome), runs one-versus-rest
statistical tests, analyzes per-observation diagnostics, and produces a narrative synthesis:

In [12]:
if RERUN:
    stat_labels = {
        'log_joint_margin': 'avg. margin between the top-2 cluster log-joints',
        'log_density': 'avg. log mixture density (typicality)',
    }

    result_df, global_results, stat_assoc_df, synthesis = explainer.explain(
        X, cluster_labels,
        y=y,
        y_label="fraudulent job posting (1=fraud, 0=legitimate)",
        observation_stats=obs_stats[['log_joint_margin', 'log_density']],
        stat_labels=stat_labels,
        synthesize=True,
        max_members_per_cluster=100,
        random_state=42,
    )

    # Save results for reproducibility
    import json
    results = {
        'result_df': result_df.to_dict(orient='records'),
        'global_results': global_results,
        'stat_assoc_df': stat_assoc_df.to_dict(orient='records'),
        'synthesis': synthesis,
    }
    with open('01_cluster_explanations.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print("Results saved to 01_cluster_explanations.json")
else:
    import json
    import pandas as pd
    with open('01_cluster_explanations.json') as f:
        results = json.load(f)
    result_df = pd.DataFrame(results['result_df'])
    global_results = results['global_results']
    stat_assoc_df = pd.DataFrame(results['stat_assoc_df'])
    synthesis = results['synthesis']
    print("Loaded pre-computed results from 01_cluster_explanations.json")

Results saved to 01_cluster_explanations.json


### Cluster descriptions and fraud associations

In [13]:
result_df

,Group Number,Short Description,Long Description,log_joint_margin,log_density,Size,Test,Odds Ratio,P-value
0,0,Financial Services & Administrative Support,Customer-facing and administrative roles in fi...,293.635724,1109.673829,1449,Fisher Exact Test,2.671455,4.693378e-22
1,1,Tech & Creative Professionals,"Mid-to-senior engineering, design, and UX role...",240.346413,1004.391040,2372,Fisher Exact Test,0.035935,8.498127e-47
2,2,Diverse Multi-Sector Workforce,Broad entry-to-executive positions spanning oi...,247.865654,969.026195,2670,Fisher Exact Test,6.953816,2.075111e-147
3,3,UK Apprenticeship Programs,Government-funded apprenticeships targeting 16...,727.426647,1508.084251,460,Fisher Exact Test,0.000000,1.553654e-10
4,4,Global Venture-Backed Startups,"High-growth startups across fintech, e-commerc...",258.148672,1019.765781,2360,Fisher Exact Test,0.234088,7.434497e-23
5,5,Early-Stage Growth Company Roles,Series A/B startups experiencing explosive gro...,209.611805,969.449652,2775,Fisher Exact Test,0.517837,4.915278e-09
6,6,Healthcare & Caregiving Services,Entry-level accessible healthcare and social s...,401.849878,1031.399335,1260,Fisher Exact Test,1.093090,4.959152e-01
7,7,English Teaching Abroad Programs,Coordinated recruitment for English teaching p...,1040.789817,1689.166385,737,Fisher Exact Test,0.000000,9.509585e-17
8,8,Enterprise Technology Infrastructure,"Mid-to-senior cloud infrastructure, DevOps, an...",233.173975,953.329636,3237,Fisher Exact Test,0.294799,1.285736e-24
9,9,"Sales, Real Estate & Logistics Operations","Entry-to-mid sales, customer service, and ware...",681.822732,1406.548430,560,Fisher Exact Test,1.239367,2.302262e-01


### Per-observation diagnostic associations

The `stat_assoc_df` shows how the GMM diagnostics (passed via `observation_stats`)
relate to the outcome:

In [14]:
stat_assoc_df

,Stat,Test,Mean (y=0),Mean (y=1),Statistic,P-value
0,log_joint_margin,Independent t-test,315.896175,252.010793,13.908964,7.032550e-41
1,log_density,Independent t-test,1053.525469,994.934480,13.663540,1.872317e-39


### Synthesis narrative

The `synthesis` output connects cluster descriptions and statistical findings
into a coherent narrative:

In [15]:
from IPython.display import Markdown
Markdown(synthesis)

# Analytical Narrative: Fraudulent Job Posting Landscape

## Overall Landscape and Cluster Themes

The ten clusters span a diverse employment ecosystem, from specialized tech roles to entry-level service positions. Three broad themes emerge: (1) **technology and innovation sectors** (Clusters 1, 4, 5, 8) emphasizing modern skills and equity compensation; (2) **accessible entry-level and service roles** (Clusters 0, 6, 7, 9) with minimal barriers to entry; and (3) **structured institutional programs** (Clusters 3, 7) with formal credentialing or government backing. Cluster 2 represents a heterogeneous staffing-agency portfolio spanning multiple sectors. This segmentation reflects fundamentally different labor market dynamics and recruitment mechanisms.

## Cluster-Level Fraud Associations: A Coherent Pattern

The fraud signal exhibits striking coherence: **technology and institutional clusters show near-zero fraud**, while **accessible entry-level clusters show elevated fraud**. 

Legitimate clusters (OR near 0) include: UK Apprenticeships (OR=0.000, p<0.0001), English Teaching Abroad (OR=0.000, p<0.0001), Tech & Creative Professionals (OR=0.036, p<0.0001), Enterprise Infrastructure (OR=0.295, p<0.0001), and Venture-Backed Startups (OR=0.234, p<0.0001). These represent 7,839 postings with exceptionally strong protective signals.

Fraudulent clusters (OR>1) include: Diverse Multi-Sector (OR=6.954, p<0.0001), Financial Services & Admin (OR=2.671, p<0.0001), and Early-Stage Growth (OR=0.518, p<0.0001). The Diverse Multi-Sector cluster shows the strongest fraud enrichment—nearly 7-fold elevated odds. Healthcare (OR=1.093, p=0.496) and Sales/Logistics (OR=1.239, p=0.230) show modest, non-significant associations.

## Per-Observation Statistics: Typicality and Clarity

Two critical per-observation metrics reveal the mechanism underlying fraud risk:

**Log joint margin** (margin between top-2 cluster assignments) shows legitimate postings average 315.9 versus fraudulent at 252.0 (t=13.909, p<0.0001). This indicates **legitimate postings are more clearly classifiable**—they fit their assigned cluster distinctly, whereas fraudulent postings blur cluster boundaries, suggesting they employ generic or misaligned language.

**Log density** (typicality within assigned cluster) shows legitimate postings average 1053.5 versus fraudulent at 994.9 (t=13.664, p<0.0001). Fraudulent postings are **less representative of their clusters**, implying they deviate from sector-specific conventions and language patterns.

These patterns are consistent across clusters: Clusters 3 and 7 (legitimate) show exceptionally high margins (727.4, 1040.8) and densities (1508.1, 1689.2), while Cluster 2 (fraudulent-enriched) shows lower margin (247.9) and density (969.0).

## Content Characteristics Distinguishing Fraud

The fraud signal correlates with **accessibility and urgency** rather than technical specificity. Fraudulent-enriched clusters emphasize:
- **Minimal qualifications** and "attitude-based hiring" (Cluster 5)
- **Staffing agency intermediation** with rapid placement (Cluster 2)
- **Bilingual/customer-facing generalism** without specialized credentials (Cluster 0)
- **Commission-based compensation** and rapid hiring (Cluster 9)

Conversely, legitimate clusters feature:
- **Specific technical requirements** (AWS/Azure, 3-7 years experience in Cluster 8)
- **Formal institutional backing** (government apprenticeships, established teaching programs)
- **Equity and mission-driven narratives** with verifiable company profiles (Clusters 1, 4)
- **Clear role specialization** and seniority levels

The pattern suggests fraudsters exploit **low-barrier-to-entry sectors** where verification is difficult and applicant desperation is highest. Legitimate postings in these sectors (e.g., Healthcare, Cluster 6) maintain specificity and institutional credibility, reducing fraud risk.

## Conclusion

Fraud is not randomly distributed but systematically concentrated in sectors combining accessibility, urgency, and staffing-agency intermediation. The strong per-observation statistics indicate fraudulent postings lack the linguistic coherence and sector-specific conventions of legitimate roles, appearing as generic templates rather than authentic opportunities. This suggests detection strategies should prioritize **typicality metrics and cluster clarity** alongside content analysis.

## 6. Predictive Pipeline

The same TabuLLM components compose directly into a scikit-learn `Pipeline` that
combines text and structured features for prediction. We first establish a non-text
baseline, then add text features via TabuLLM.

The structured features require different preprocessing strategies:
- **Binary** (3 columns): passed through unchanged
- **Low-cardinality categorical** (3 columns: employment type, experience level, education level): one-hot encoded, with missing values treated as a distinct category
- **Medium-cardinality categorical** (2 columns: industry, job function): target-encoded to avoid the dimensionality explosion of one-hot encoding

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Column groups
binary_cols = metadata['binary_columns']
low_card_cols = ['employment_type', 'required_experience', 'required_education']
med_card_cols = ['industry', 'function']

### Baseline: non-text features only

In [17]:
baseline_pipeline = Pipeline([
    ('features', ColumnTransformer([
        ('binary', 'passthrough', binary_cols),
        ('low_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), low_card_cols),
        ('med_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('target', TargetEncoder(cv=5, smooth='auto', target_type='binary',
                                     random_state=42))
        ]), med_card_cols),
    ])),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
baseline_auc = roc_auc_score(
    y_test, baseline_pipeline.predict_proba(X_test)[:, 1]
)
print(f"Baseline (non-text features only) Test AUC: {baseline_auc:.4f}")

Baseline (non-text features only) Test AUC: 0.9157


### Full pipeline: text + structured features

In [ ]:
full_pipeline = Pipeline([
    ('features', ColumnTransformer([
        ('text', Pipeline([
            ('embed', TextColumnTransformer(model=embedding_model)),
            ('reduce', GMMFeatureExtractor(n_components=10,
                                           include_log_density=True,
                                           random_state=42))
        ]), text_cols),
        ('binary', 'passthrough', binary_cols),
        ('low_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), low_card_cols),
        ('med_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('target', TargetEncoder(cv=5, smooth='auto', target_type='binary',
                                     random_state=42))
        ]), med_card_cols),
    ])),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_train, y_train)
full_auc = roc_auc_score(
    y_test, full_pipeline.predict_proba(X_test)[:, 1]
)
print(f"Full pipeline (text + structured) Test AUC: {full_auc:.4f}")
print(f"Improvement over baseline: +{full_auc - baseline_auc:.4f}")

## Next Steps

This notebook demonstrated the core TabuLLM workflow: embedding, dimensionality reduction,
cluster explanation, and predictive pipeline integration.

For advanced pipeline patterns — including column selection analysis and stacking ensembles
that push AUC further — see `02_advanced_pipelines.ipynb`.